# Optional Full-LWC Benchmarks
Full-cell enumeration can grow exponentially with dimension and calibration size. This notebook is deliberately disabled by default and uses one worker. Existing results are preserved; a separate output folder holds manual runs. Each trial generates fresh training, calibration, and test observations, refits OLS, and verifies pairing with the saved envelope cohort.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'envelope_method':
    ROOT = ROOT.parent
assert (ROOT / 'utility').is_dir(), 'Open this notebook from the project root or envelope_method.'
sys.path[:0] = [str(ROOT), str(ROOT / 'envelope_method')]
import importlib.util
if importlib.util.find_spec('sklearn') is None and (ROOT / 'tmp/diagnostic_packages').exists():
    sys.path.insert(0, str(ROOT / 'tmp/diagnostic_packages'))
import pandas as pd
from run_full_lwc_scaling import run
inventory = json.loads((ROOT / 'envelope_method/settings.json').read_text())


In [ ]:
RUN_FULL_LWC = False
DIMENSIONS = [2]
CALIBRATION_SIZES = [10]
MAX_TRIALS = 10
INCLUDE_EXACT_RATIO = True
OUTPUT_FAMILY = 'full_lwc_manual'
# Original scaling studies: n=10, d=2..6; n=30, d=2..4.
# Original 2D sample-size comparison: n=30,50,100,300,500.
selected = [i for i in inventory if i['config']['kind'] == 'absolute'
            and i['config']['noise_type'] == 'Laplace'
            and i['config']['n_train'] == 6400
            and i['config']['d'] in DIMENSIONS
            and i['config']['n_cal'] in CALIBRATION_SIZES]
assert selected, 'No matching saved envelope cohort.'
pd.DataFrame([dict(config_id=i['id'], d=i['config']['d'], n_cal=i['config']['n_cal'],
                   trials=min(MAX_TRIALS,i['trials']),
                   worst_case_cells=(i['config']['n_cal']+1)**i['config']['d']) for i in selected])


In [ ]:
if RUN_FULL_LWC:
    for item in selected:  # Deliberately sequential, one worker.
        print('Running', item['id'], flush=True)
        run(item, trials=min(MAX_TRIALS,item['trials']), output_family=OUTPUT_FAMILY,
            exact_variants=(False, True) if INCLUDE_EXACT_RATIO else (False,))
else:
    print('No computation started. Set RUN_FULL_LWC=True to run the selected configurations.')


In [ ]:
tables = []
for item in selected:
    path = ROOT / 'envelope_method/results' / OUTPUT_FAMILY / item['id'] / 'trials.csv'
    if not path.exists():
        continue
    local = pd.read_csv(path)
    primary = pd.read_csv(ROOT / 'envelope_method/results/absolute' / item['id'] / 'trials.csv')
    primary = primary[primary.trial.isin(local.trial) & primary.method.isin(['Envelope','TSCP_R','TSCP_GWC'])]
    tables.append(pd.concat([local,primary],ignore_index=True))
if tables:
    results = pd.concat(tables,ignore_index=True)
    display(results.groupby(['config_id','method'])[['test_coverage','outcome_volume','runtime']].agg(['mean','std','count']))
